# AEMO DUID Network Constraint Forecast Demo

This notebook is the compact Phase 2.5 story for the `CATHROCK` 6h horizon demo. It is designed for review: target choice, label definition, model metrics, rare-event backtesting, API usage, and limitations.

## 0. Setup

Run from the repository root after installing `requirements.txt`. The notebook reads the generated Phase 2/2.5 artifacts when they exist, so reviewers can inspect the demo without rerunning every download.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.aemo_constraint_demo.aemo_client import read_monthly_table
from src.aemo_constraint_demo.features import relevant_constraints_for_duid

DUID = "CATHROCK"
DAYS = 60
HORIZON_HOURS = 6
TARGET_COLUMN = "target_constrained_next_6h"
CACHE_DIR = REPO_ROOT / "data" / "cache"
MODEL_PATH = REPO_ROOT / "artifacts" / "cathrock_6h.joblib"
METRICS_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_metrics.json"
BACKTEST_SUMMARY_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_backtest_summary.json"
BACKTEST_FOLDS_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_backtest_folds.csv"

## 1. Target discovery summary

Phase 1.5 expanded target discovery showed `CATHROCK` had usable sparse positives for a future-only 6h horizon label. The selected configuration is deliberately narrow: one DUID, 60 recent days, and one auditable horizon target.

In [ ]:
scan_path = REPO_ROOT / "artifacts" / "duid_constraint_scan_expanded_top.csv"
if scan_path.exists():
    scan = pd.read_csv(scan_path)
    selected = scan[(scan["duid"] == DUID) & (scan["days"] == DAYS) & (scan["horizon_hours"] == HORIZON_HOURS)]
    display(selected.head(1)[[
        "duid",
        "days",
        "horizon_hours",
        "connection_point_id",
        "relevant_genconid_count",
        "positive_intervals",
        "positive_rate",
        "recommended",
    ]])
else:
    print("Run scripts/find_constrained_duids.py to regenerate target discovery artifacts.")

## 2. Map DUID -> connection point -> GENCONID

The domain mapping is `DUID -> CONNECTIONPOINTID -> GENCONID`. The model target only uses generic constraints relevant to the selected connection point.

In [ ]:
dudetail = read_monthly_table("DUDETAIL", cache_dir=CACHE_DIR)
spd_cp = read_monthly_table("SPDCONNECTIONPOINTCONSTRAINT", cache_dir=CACHE_DIR)
# Planned/invoked constraint response support uses these standing-data tables in the training bundle.
genconset = read_monthly_table("GENCONSET", cache_dir=CACHE_DIR)
genconsetinvoke = read_monthly_table("GENCONSETINVOKE", cache_dir=CACHE_DIR)

relevant_genconids, connection_point, lhs_rows = relevant_constraints_for_duid(DUID, dudetail, spd_cp)
print({
    "duid": DUID,
    "connection_point": connection_point,
    "relevant_genconids": len(relevant_genconids),
})
lhs_rows.head()

## 3. Horizon label definition

For each interval `t`, `target_constrained_next_6h` is `1` when any relevant constraint binds in a future dispatch interval from `t+1` through `t+6h`. The current interval is excluded, and incomplete tail rows are dropped during training.

In [ ]:
print(f"Target: {TARGET_COLUMN}")
print("Current interval is excluded from the future horizon label.")

## 4. Training command

The Phase 2 model is intentionally small and explainable: balanced logistic regression with time and weather features.

In [ ]:
training_command = """python -m src.aemo_constraint_demo.train \
  --duid CATHROCK \
  --days 60 \
  --horizon-hours 6 \
  --model-out artifacts/cathrock_6h.joblib"""
print(training_command)

## 5. Model metrics

The chronological holdout contains no positive `CATHROCK` 6h labels, so AP and ROC AUC are `null`. This is a data limitation to report honestly, not a result to hide.

In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text())
    keys = [
        "duid",
        "days",
        "horizon_hours",
        "target_column",
        "model_type",
        "total_samples",
        "train_samples",
        "test_samples",
        "positive_rate",
        "test_positive_rate",
        "average_precision",
        "roc_auc",
        "precision_at_top_5_percent",
    ]
    display(pd.DataFrame([{key: metrics.get(key) for key in keys}]).T.rename(columns={0: "value"}))
else:
    print("Run the training command to create artifacts/cathrock_6h_metrics.json.")

## 6. Event-window backtest summary

Phase 2.5 adds event-window folds. Each fold trains only on rows strictly before the validation window and skips invalid one-class training or validation windows.

In [ ]:
backtest_command = """python scripts/backtest_horizon_model.py \
  --duid CATHROCK \
  --days 60 \
  --horizon-hours 6 \
  --model-out artifacts/cathrock_6h.joblib \
  --out artifacts/cathrock_6h_backtest_summary.json \
  --folds-out artifacts/cathrock_6h_backtest_folds.csv \
  --max-folds 5"""
print(backtest_command)

if BACKTEST_SUMMARY_PATH.exists():
    summary = json.loads(BACKTEST_SUMMARY_PATH.read_text())
    summary_keys = [
        "evaluated_folds",
        "skipped_folds",
        "mean_average_precision",
        "mean_roc_auc",
        "mean_precision_at_top_5_percent",
        "mean_lift_at_top_5_percent",
    ]
    display(pd.DataFrame([{key: summary.get(key) for key in summary_keys}]).T.rename(columns={0: "value"}))
if BACKTEST_FOLDS_PATH.exists():
    display(pd.read_csv(BACKTEST_FOLDS_PATH)[[
        "fold_id",
        "validation_start",
        "validation_end",
        "train_samples",
        "validation_samples",
        "train_positive_rate",
        "validation_positive_rate",
        "average_precision",
        "roc_auc",
        "skipped",
        "skip_reason",
    ]])

## 7. API query examples

The API infers `horizon_hours=6` from the model bundle when the query parameter is omitted. Explicit `horizon_hours=6` is also supported; mismatched horizons return a clear 400 error.

In [ ]:
print('uvicorn app:app --reload')
print('curl "http://127.0.0.1:8000/forecast/CATHROCK?model_path=artifacts/cathrock_6h.joblib"')
print('curl "http://127.0.0.1:8000/forecast/CATHROCK?horizon_hours=6&model_path=artifacts/cathrock_6h.joblib"')

example_path = REPO_ROOT / "artifacts" / "example_cathrock_6h_response.json"
if example_path.exists():
    example = json.loads(example_path.read_text())
    print({
        "duid": example["duid"],
        "horizon_hours": example["horizon_hours"],
        "forecast_rows": len(example["forecast"]),
        "first_probability": example["forecast"][0]["probability"],
    })

## 8. Limitations

This is a transparent workflow demo, not a production-accurate forecaster. The chronological holdout has no positive labels, event-window folds are sparse and partly overlapping, top-k lift is mixed, and production work would need longer history, P5MIN/Predispatch inputs, outage context, constraint taxonomy, calibration, and monitoring.